# Week 6: Model Evaluation — Is Our Model Actually Any Good?

**Where we are.** Last week we walked through the full ML lifecycle and trained our first model. This week we zoom in on **Stage 5: Evaluation**, the stage that decides whether a model is trustworthy enough to use.

**Today's roadmap:**

| Step | Topic |
|---|---|
| 0 | Recap: get back to a trained model |
| 1 | The confusion matrix |
| 2 | Accuracy, and why it can mislead |
| 3 | Precision, recall, and F1 |
| 4 | Thresholds: trading one kind of mistake for another |
| 5 | Overfitting: when a model memorizes instead of learns |
| 6 | Tuning: improving a model without cheating |
| 7 | Comparing models fairly |
| 8 | **In-class activity (IC-5):** Should we use AI? |

**How this notebook works (same as last week).** For each step I'll **demo** the idea on a sample dataset. Then you'll see a **🔧 Your Turn** section where you repeat the step on your own dataset.

> **One change from last week:** everyone now uses the **same dataset** in the Your Turn sections (loaded for you in Step 0), so we can compare results and discuss them together. Just run the cells top to bottom.

Still no neural networks. We're using classical models: logistic regression, decision trees, random forests, and KNN.

In [ ]:
# Run this cell first — it loads the libraries we'll use all notebook long.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 3)
print('Libraries loaded.')

---
## Step 0 — Recap: Get Back to a Trained Model

Last week's steps, condensed into one cell each: **load → split → train**. If any of this looks unfamiliar, review last week's notebook first.

**Demo dataset:** the breast-cancer diagnostic data from scikit-learn (same as last week).

> ⚠️ **One fix from last week.** In scikit-learn's version of this data, `0 = malignant` and `1 = benign`. That made "positive" mean *benign*, the opposite of what we care about catching. This week we **flip it so `1 = malignant`**. Now a "positive" is the thing we're trying to catch, which makes precision and recall much easier to read.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df_demo = data.frame.copy()

# Flip the labels so that 1 = malignant (the case we want to catch)
df_demo['target'] = 1 - df_demo['target']

print('Shape:', df_demo.shape)
print('\nTarget balance (1 = malignant):')
print(df_demo['target'].value_counts(normalize=True))

In [ ]:
# Split (stratified, as before) and train a logistic regression
X_demo = df_demo.drop(columns=['target'])
y_demo = df_demo['target']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42, stratify=y_demo
)

model_demo = LogisticRegression(max_iter=5000)
model_demo.fit(X_train_d, y_train_d)
preds_demo = model_demo.predict(X_test_d)

print('Train size:', X_train_d.shape, ' Test size:', X_test_d.shape)
print('Demo model trained. Example predictions:', preds_demo[:10])

### 🔧 Your Turn — Load your dataset and train your model

**Your dataset: Pima Indians Diabetes.** Each row is a woman (21+ years old) with medical measurements, and the target is `Outcome`: **1 = diagnosed with diabetes, 0 = not**.

**Your scenario:** a community clinic wants a model that flags patients who should get a follow-up diabetes test. The **positive class (1) is "has diabetes"**, the thing we want to catch.

The file has no header row, so the code below adds column names for you. No files to upload.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df_yours = pd.read_csv(url, header=None, names=columns)

print('Shape:', df_yours.shape)
print('\nTarget balance (1 = diabetes):')
print(df_yours['Outcome'].value_counts(normalize=True))
df_yours.head()

In [ ]:
# Split (stratified) and train a logistic regression
X_yours = df_yours.drop(columns=['Outcome'])
y_yours = df_yours['Outcome']

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    X_yours, y_yours, test_size=0.2, random_state=42, stratify=y_yours
)

model_yours = LogisticRegression(max_iter=5000)
model_yours.fit(X_train_y, y_train_y)
preds_yours = model_yours.predict(X_test_y)

print('Train size:', X_train_y.shape, ' Test size:', X_test_y.shape)
print('Your model trained. Example predictions:', preds_yours[:10])

> 📝 **Data note.** In this dataset, some columns (like `Insulin` and `SkinThickness`) use `0` to mean "not measured." A careful analyst would clean that up. We're leaving it as-is to keep today focused on *evaluation*. Keep it in mind when you interpret your results.

---
## Step 1 — The Confusion Matrix

Every classification metric we'll use today comes from one small table: the **confusion matrix**. It counts how often the model was right or wrong, *and in which direction*.

|  | **Predicted: Negative (0)** | **Predicted: Positive (1)** |
|---|---|---|
| **Actually Negative (0)** | **True Negative (TN)** ✅ correctly cleared | **False Positive (FP)** ❌ false alarm |
| **Actually Positive (1)** | **False Negative (FN)** ❌ missed case | **True Positive (TP)** ✅ correctly caught |

The two ✅ cells are correct predictions. The two ❌ cells are the two *different kinds of mistakes*, and they rarely cost the same.

**Demo:** for the tumor model, a **positive** means "predicted malignant."

In [ ]:
cm = confusion_matrix(y_test_d, preds_demo)
tn, fp, fn, tp = cm.ravel()

print(f'True Negatives  (TN): {tn:>3}  benign, correctly cleared')
print(f'False Positives (FP): {fp:>3}  benign, wrongly flagged as malignant  (false alarm)')
print(f'False Negatives (FN): {fn:>3}  malignant, wrongly cleared            (MISSED case)')
print(f'True Positives  (TP): {tp:>3}  malignant, correctly caught')

ConfusionMatrixDisplay.from_predictions(
    y_test_d, preds_demo,
    display_labels=['Benign', 'Malignant'], cmap='Blues'
)
plt.title('Confusion Matrix — Demo Model (tumors)')
plt.show()

**Read it out loud in plain English.** For the demo: *"Out of every malignant tumor in the test set, the model caught \_\_ and missed \_\_. Out of every benign tumor, it wrongly flagged \_\_."* Being able to say this sentence is the skill. The numbers are just details.

### 🔧 Your Turn — Read your confusion matrix

In [ ]:
cm_y = confusion_matrix(y_test_y, preds_yours)
tn_y, fp_y, fn_y, tp_y = cm_y.ravel()

print(f'True Negatives  (TN): {tn_y:>3}  no diabetes, correctly cleared')
print(f'False Positives (FP): {fp_y:>3}  no diabetes, wrongly flagged           (false alarm)')
print(f'False Negatives (FN): {fn_y:>3}  HAS diabetes, wrongly cleared           (MISSED case)')
print(f'True Positives  (TP): {tp_y:>3}  has diabetes, correctly caught')

ConfusionMatrixDisplay.from_predictions(
    y_test_y, preds_yours,
    display_labels=['No diabetes', 'Diabetes'], cmap='Blues'
)
plt.title('Confusion Matrix — Your Model (diabetes)')
plt.show()

**Your answers:**

1. Write the plain-English sentence for your matrix (caught / missed / false alarms):

2. In a diabetes screening clinic, what actually happens to a patient who is a **false negative**? What about a **false positive**? Which is worse?

---
## Step 2 — Accuracy, and Why It Can Mislead

**Accuracy** = (TN + TP) / everything. The share of predictions that were correct. It's easy to explain, and it's easy to be fooled by.

**The test:** let's compare our model against a "model" that does **no learning at all**. It just predicts the most common class for everyone (scikit-learn's `DummyClassifier`). If our real model can't clearly beat this baseline, it hasn't learned anything useful.

In [ ]:
dummy_d = DummyClassifier(strategy='most_frequent')
dummy_d.fit(X_train_d, y_train_d)
dummy_preds_d = dummy_d.predict(X_test_d)

comparison = pd.DataFrame({
    'Accuracy': [accuracy_score(y_test_d, dummy_preds_d), accuracy_score(y_test_d, preds_demo)],
    'Recall (malignant caught)': [recall_score(y_test_d, dummy_preds_d),
                                  recall_score(y_test_d, preds_demo)],
}, index=['"Always predict benign" (no learning)', 'Logistic regression'])

comparison

Notice the "always predict benign" strategy scores a respectable-looking accuracy while **catching zero malignant tumors**. Accuracy hides that, because the majority class dominates the count.

> **Rule of thumb:** always ask "**what accuracy would a do-nothing baseline get?**" Beating 50% means nothing if 65% of the data is one class. This matters most when your target is **imbalanced**.

### 🔧 Your Turn — Beat the baseline?

In [ ]:
dummy_y = DummyClassifier(strategy='most_frequent')
dummy_y.fit(X_train_y, y_train_y)
dummy_preds_y = dummy_y.predict(X_test_y)

comparison_y = pd.DataFrame({
    'Accuracy': [accuracy_score(y_test_y, dummy_preds_y), accuracy_score(y_test_y, preds_yours)],
    'Recall (diabetes caught)': [recall_score(y_test_y, dummy_preds_y),
                                 recall_score(y_test_y, preds_yours)],
}, index=['"Always predict no diabetes" (no learning)', 'Logistic regression'])

comparison_y

**Your answers:**

1. How much better is your model's accuracy than the do-nothing baseline? Does that gap feel impressive?

2. Compare the two **recall** values. What does the recall column tell you that accuracy didn't?

---
## Step 3 — Precision, Recall, and F1

These three metrics look at the **positive class** (the thing we're trying to catch):

| Metric | Formula | The question it answers |
|---|---|---|
| **Precision** | TP / (TP + FP) | *"When the model raises an alarm, how often is it right?"* |
| **Recall** | TP / (TP + FN) | *"Of all the real cases, how many did the model catch?"* |
| **F1** | 2 × (P × R) / (P + R) | *"One number balancing both."* It's low if **either** precision or recall is low. |

**Which matters more depends on the cost of each mistake:**
- **High recall matters** when *missing* a case is costly (cancer screening, fraud detection, safety defects).
- **High precision matters** when *false alarms* are costly (spam filters that hide real email, flagging innocent customers, wasting an expert's time).

**Demo:** first calculate them by hand from the confusion matrix, then confirm scikit-learn agrees.

In [ ]:
# By hand, straight from the confusion matrix counts (tn, fp, fn, tp from Step 1)
precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)

print('--- By hand ---')
print(f'Precision: {precision_manual:.3f}   ({tp} / ({tp} + {fp}))')
print(f'Recall:    {recall_manual:.3f}   ({tp} / ({tp} + {fn}))')
print(f'F1:        {f1_manual:.3f}')

# scikit-learn's version -- should match
print('\n--- scikit-learn ---')
print(f'Accuracy:  {accuracy_score(y_test_d, preds_demo):.3f}')
print(f'Precision: {precision_score(y_test_d, preds_demo):.3f}')
print(f'Recall:    {recall_score(y_test_d, preds_demo):.3f}')
print(f'F1:        {f1_score(y_test_d, preds_demo):.3f}')

print('\n--- Full report ---')
print(classification_report(y_test_d, preds_demo, target_names=['Benign', 'Malignant']))

### 🔧 Your Turn — Precision, recall, and F1 for your model

In [ ]:
precision_manual_y = tp_y / (tp_y + fp_y)
recall_manual_y = tp_y / (tp_y + fn_y)
f1_manual_y = 2 * precision_manual_y * recall_manual_y / (precision_manual_y + recall_manual_y)

print('--- By hand ---')
print(f'Precision: {precision_manual_y:.3f}   ({tp_y} / ({tp_y} + {fp_y}))')
print(f'Recall:    {recall_manual_y:.3f}   ({tp_y} / ({tp_y} + {fn_y}))')
print(f'F1:        {f1_manual_y:.3f}')

print('\n--- scikit-learn ---')
print(f'Accuracy:  {accuracy_score(y_test_y, preds_yours):.3f}')
print(f'Precision: {precision_score(y_test_y, preds_yours):.3f}')
print(f'Recall:    {recall_score(y_test_y, preds_yours):.3f}')
print(f'F1:        {f1_score(y_test_y, preds_yours):.3f}')

print('\n--- Full report ---')
print(classification_report(y_test_y, preds_yours, target_names=['No diabetes', 'Diabetes']))

**Your answers:**

1. In one sentence each, explain what your model's precision and recall *mean* for a clinic patient.

2. For a diabetes screening program, would you rather have higher precision or higher recall? Why?

---
## Step 4 — Thresholds: Trading One Mistake for Another

Most classifiers don't just say "yes/no." They output a **probability**, and `.predict()` quietly applies a **threshold of 0.5**: *"if the probability of class 1 is ≥ 0.5, say positive."*

But 0.5 is a default, not a law. We can move it:

- **Lower the threshold** → flag more cases → **recall goes up** (fewer misses), **precision goes down** (more false alarms).
- **Raise the threshold** → flag fewer cases → **precision goes up**, **recall goes down**.

You can't get both for free. Choosing the threshold is a **business decision** about which mistake you can better afford.

In [ ]:
def threshold_table(y_true, probs, thresholds=np.arange(0.1, 1.0, 0.1)):
    """For each threshold, report the metrics and the raw counts of each mistake."""
    y_true = np.asarray(y_true)
    rows = []
    for t in thresholds:
        pred = (probs >= t).astype(int)
        rows.append({
            'threshold': round(float(t), 2),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'missed (FN)': int(((pred == 0) & (y_true == 1)).sum()),
            'false alarms (FP)': int(((pred == 1) & (y_true == 0)).sum()),
        })
    return pd.DataFrame(rows).set_index('threshold')

# Probability of class 1 (malignant) for each test tumor
probs_demo = model_demo.predict_proba(X_test_d)[:, 1]

threshold_table(y_test_d, probs_demo)

In [ ]:
# Visualize the trade-off across many thresholds
fine = np.linspace(0.01, 0.99, 99)
tt = threshold_table(y_test_d, probs_demo, fine)

plt.figure(figsize=(7, 4))
plt.plot(tt.index, tt['precision'], label='Precision')
plt.plot(tt.index, tt['recall'], label='Recall')
plt.axvline(0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs. Recall as the threshold moves — Demo model')
plt.legend()
plt.show()

> ⚠️ **Heads-up for real projects:** we're exploring on the *test set* here so we can see the trade-off. In practice, you'd choose your threshold using **validation/cross-validation data** and keep the test set for one final honest check. Choosing thresholds by peeking at the test set makes the results look better than they'd really be. (We'll come back to this idea in Step 6.)

### 🔧 Your Turn — Find a threshold for your clinic

Now that you have the trade-off table, let's add **costs**. Suppose a **missed diabetes case (FN)** is 5× worse than a **false alarm (FP)**, because an unnecessary follow-up test is cheap compared with an undiagnosed patient going untreated.

In [ ]:
probs_yours = model_yours.predict_proba(X_test_y)[:, 1]   # probability of diabetes

table_y = threshold_table(y_test_y, probs_yours)

# TODO: try changing these costs and re-running to see the best threshold move
cost_fn = 5   # cost of MISSING a real diabetes case
cost_fp = 1   # cost of a false alarm (unnecessary follow-up)

table_y['total_cost'] = table_y['missed (FN)'] * cost_fn + table_y['false alarms (FP)'] * cost_fp
display(table_y)

best_t = table_y['total_cost'].idxmin()
print(f'\nLowest total cost at threshold = {best_t}')

In [ ]:
# Plot the trade-off for your model
fine = np.linspace(0.01, 0.99, 99)
tt_y = threshold_table(y_test_y, probs_yours, fine)

plt.figure(figsize=(7, 4))
plt.plot(tt_y.index, tt_y['precision'], label='Precision')
plt.plot(tt_y.index, tt_y['recall'], label='Recall')
plt.axvline(0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
plt.axvline(best_t, color='green', linestyle=':', label=f'Lowest-cost threshold ({best_t})')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs. Recall as the threshold moves — Your model')
plt.legend()
plt.show()

**Your answers:**

1. What threshold minimized total cost, and what happened to recall and precision at that threshold compared with 0.5?

2. Change `cost_fn` to `1` (missing a case costs the same as a false alarm) and re-run. Where does the best threshold move? What does that tell you about how costs drive the decision?

---
## Step 5 — Overfitting: When a Model Memorizes Instead of Learns

A model can look great on the data it trained on and still fall apart on new data. That's **overfitting**: it memorized quirks and noise in the training set instead of learning the general pattern.

| | Training score | Score on unseen data | What's happening |
|---|---|---|---|
| **Underfitting** | Low | Low | Model is too simple to capture the pattern |
| **Good fit** | High | About as high | It learned something that generalizes |
| **Overfitting** | Very high | Noticeably lower | It memorized the training data |

**The warning sign is the gap between training and unseen performance.** That's why we never judge a model on the data it trained on.

**Decision trees are the classic example.** An unrestricted tree can keep splitting until it has memorized every training row.

In [ ]:
# An unrestricted decision tree: no limit on how deep it can grow
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train_d, y_train_d)

print(f'Tree depth it grew: {tree_full.get_depth()}')
print(f'F1 on TRAINING data: {f1_score(y_train_d, tree_full.predict(X_train_d)):.3f}')
print(f'F1 on TEST data:     {f1_score(y_test_d, tree_full.predict(X_test_d)):.3f}')

A perfect (or near-perfect) training score with a lower test score is a red flag. **Can we control it?** Yes. Limiting the tree's `max_depth` forces it to learn simpler patterns. Let's see how training score and *unseen-data* score behave as the tree gets deeper.

To judge "unseen-data" performance **without touching the test set**, we use **cross-validation (CV)**: split the *training* data into 5 chunks, train on 4, score on the 5th, rotate through all five, and average. Each score comes from data that the model wasn't trained on.

In [ ]:
depths = range(1, 16)
train_f1, cv_f1 = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_d, y_train_d)
    train_f1.append(f1_score(y_train_d, tree.predict(X_train_d)))
    cv_f1.append(cross_val_score(tree, X_train_d, y_train_d, cv=5, scoring='f1').mean())

plt.figure(figsize=(7, 4))
plt.plot(depths, train_f1, marker='o', label='Training F1')
plt.plot(depths, cv_f1, marker='o', label='Cross-validated F1 (unseen folds)')
plt.xlabel('Tree depth (model complexity)')
plt.ylabel('F1 score')
plt.title('Overfitting curve — Demo (decision tree)')
plt.legend()
plt.show()

**How to read it:** as depth grows, the training line climbs toward 1.0 while the cross-validated line flattens or drops. The widening gap *is* overfitting. The "sweet spot" is roughly where the cross-validated score peaks, not where the training score is highest.

### 🔧 Your Turn — Watch your tree overfit

In [ ]:
tree_full_y = DecisionTreeClassifier(random_state=42)
tree_full_y.fit(X_train_y, y_train_y)

print(f'Tree depth it grew: {tree_full_y.get_depth()}')
print(f'F1 on TRAINING data: {f1_score(y_train_y, tree_full_y.predict(X_train_y)):.3f}')
print(f'F1 on TEST data:     {f1_score(y_test_y, tree_full_y.predict(X_test_y)):.3f}')

In [ ]:
depths = range(1, 16)
train_f1_y, cv_f1_y = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_y, y_train_y)
    train_f1_y.append(f1_score(y_train_y, tree.predict(X_train_y)))
    cv_f1_y.append(cross_val_score(tree, X_train_y, y_train_y, cv=5, scoring='f1').mean())

plt.figure(figsize=(7, 4))
plt.plot(depths, train_f1_y, marker='o', label='Training F1')
plt.plot(depths, cv_f1_y, marker='o', label='Cross-validated F1 (unseen folds)')
plt.xlabel('Tree depth (model complexity)')
plt.ylabel('F1 score')
plt.title('Overfitting curve — Your data (decision tree)')
plt.legend()
plt.show()

**Your answers:**

1. How big is the training-vs-test gap for the unrestricted tree on your data? Is it bigger or smaller than the demo's?

2. Around what depth does the cross-validated F1 peak? Beyond that depth, what happens to training F1 versus cross-validated F1?

---
## Step 6 — Tuning: Improving a Model Without Cheating

Models have settings you choose *before* training, called **hyperparameters** (like a tree's `max_depth`). **Tuning** means trying different settings to find the best combination.

**The trap:** if you try 50 settings and pick whichever scores best on the *test set*, you've indirectly trained on the test set. It's no longer honest. The correct workflow:

1. **Tune using cross-validation on the training data only.**
2. **Pick the best settings.**
3. **Touch the test set once**, at the very end, for the final score you report.

scikit-learn's `GridSearchCV` automates steps 1–2: it tries every combination in a grid, scores each one with cross-validation, and keeps the best.

First, a small helper we'll reuse for the rest of the notebook. It reports a model's train and test performance side by side, which makes overfitting visible.

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    """Return test-set metrics plus the train-vs-test F1 gap for one fitted model."""
    test_pred = model.predict(X_te)
    train_f1 = f1_score(y_tr, model.predict(X_tr))
    test_f1 = f1_score(y_te, test_pred)
    return {
        'Accuracy': accuracy_score(y_te, test_pred),
        'Precision': precision_score(y_te, test_pred, zero_division=0),
        'Recall': recall_score(y_te, test_pred, zero_division=0),
        'F1': test_f1,
        'Train F1': train_f1,
        'Overfit gap (Train F1 - Test F1)': train_f1 - test_f1,
    }
print('Helper defined.')

In [ ]:
# Tune a decision tree with 5-fold cross-validation on the TRAINING data only
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20],
}

grid_d = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='f1'
)
grid_d.fit(X_train_d, y_train_d)

print('Best settings found:', grid_d.best_params_)
print(f'Best cross-validated F1: {grid_d.best_score_:.3f}')

In [ ]:
# NOW we touch the test set, once: untuned tree vs. tuned tree
tree_tuned_d = grid_d.best_estimator_

before_after = pd.DataFrame({
    'Untuned tree': evaluate_model(tree_full, X_train_d, y_train_d, X_test_d, y_test_d),
    'Tuned tree':   evaluate_model(tree_tuned_d, X_train_d, y_train_d, X_test_d, y_test_d),
}).T
before_after

Look at the last column: tuning should have **shrunk the overfit gap**, and often nudged test performance up too. Not every tuning run beats the default, and that's fine. What matters is that the process was honest.

> 💡 **`scoring='f1'` was a choice.** GridSearchCV picks the "best" settings by whatever score you tell it to. If missing cases is what you can't afford, you might tune for `'recall'` instead. The metric you optimize is a business decision, not a technical default.

### 🔧 Your Turn — Tune your tree

In [ ]:
grid_y = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='f1'      # TODO: try scoring='recall' and compare!
)
grid_y.fit(X_train_y, y_train_y)

print('Best settings found:', grid_y.best_params_)
print(f'Best cross-validated F1: {grid_y.best_score_:.3f}')

In [ ]:
tree_tuned_y = grid_y.best_estimator_

before_after_y = pd.DataFrame({
    'Untuned tree': evaluate_model(tree_full_y, X_train_y, y_train_y, X_test_y, y_test_y),
    'Tuned tree':   evaluate_model(tree_tuned_y, X_train_y, y_train_y, X_test_y, y_test_y),
}).T
before_after_y

**Your answers:**

1. What settings did the grid search pick? Did tuning shrink the overfit gap?

2. Why is it important that we tuned with cross-validation on the *training* data rather than picking settings by looking at test-set scores?

---
## Step 7 — Comparing Models Fairly

There's rarely one "right" model, so standard practice is to train several and compare them on the **same test set with the same metrics**. A fair comparison means:

- Same train/test split for every model
- Multiple metrics (not just accuracy)
- Checking the **overfit gap**, not just test scores

**Demo:** we'll compare our logistic regression, our tuned decision tree, and a **random forest**. A random forest trains many trees on random subsets of the data and lets them vote, which usually generalizes better than a single unrestricted tree.

> 👀 **Watch out:** a random forest often scores ~1.0 on its *own training data* even when it performs well on new data, so its overfit gap can look large. Judge models by their **test scores first**, and use the gap as a warning sign rather than a verdict.

In [ ]:
def compare_models(models, X_tr, y_tr, X_te, y_te):
    """Evaluate a dict of {name: fitted_model} and return a comparison table."""
    rows = {name: evaluate_model(m, X_tr, y_tr, X_te, y_te) for name, m in models.items()}
    return pd.DataFrame(rows).T

def plot_comparison(results, title):
    """Grouped bar chart of precision / recall / F1 for each model."""
    results[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(8, 4), rot=0)
    plt.ylim(0, 1.05)
    plt.ylabel('Score')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.show()

# Train the random forest for the demo
forest_d = RandomForestClassifier(n_estimators=200, random_state=42)
forest_d.fit(X_train_d, y_train_d)

models_demo = {
    'Logistic Regression': model_demo,
    'Decision Tree (tuned)': tree_tuned_d,
    'Random Forest': forest_d,
}

results_demo = compare_models(models_demo, X_train_d, y_train_d, X_test_d, y_test_d)
display(results_demo)
plot_comparison(results_demo, 'Model comparison — Demo (tumors)')

### 🔧 Your Turn — Compare models on your data

In [ ]:
forest_y = RandomForestClassifier(n_estimators=200, random_state=42)
forest_y.fit(X_train_y, y_train_y)

models_yours = {
    'Logistic Regression': model_yours,
    'Decision Tree (tuned)': tree_tuned_y,
    'Random Forest': forest_y,
}

results_yours = compare_models(models_yours, X_train_y, y_train_y, X_test_y, y_test_y)
display(results_yours)
plot_comparison(results_yours, 'Model comparison — Your data (diabetes)')

**Your answers:**

1. Which model has the best **recall**? Best **precision**? Best **F1**? Is it the same model each time?

2. Which model shows the biggest overfit gap? Does the model with the best training score also have the best test score?

---
## Step 8 — In-Class Activity (IC-5): Should We Use AI?

A good evaluation ends in a **decision**, and sometimes the right decision is *not* to use ML at all. Recall Week 1 (AI vs. automation): a simple rule can be cheaper, more transparent, and just as good.

### The case
The community clinic wants to use your diabetes model at check-in to flag patients for a follow-up blood test. The alternative they already know is a **simple rule**: *"flag anyone whose glucose reading is 120 or higher."* No ML required.

**Is the ML model actually better than the rule?** Let's test it on the same test set.

In [ ]:
# The "no-AI" alternative: a simple rule based on one column
rule_preds = (X_test_y['Glucose'] >= 120).astype(int)

rule_vs_model = pd.DataFrame({
    'Simple rule (Glucose >= 120)': {
        'Accuracy': accuracy_score(y_test_y, rule_preds),
        'Precision': precision_score(y_test_y, rule_preds),
        'Recall': recall_score(y_test_y, rule_preds),
        'F1': f1_score(y_test_y, rule_preds),
    },
    'Logistic Regression': results_yours.loc['Logistic Regression', ['Accuracy', 'Precision', 'Recall', 'F1']],
    'Random Forest': results_yours.loc['Random Forest', ['Accuracy', 'Precision', 'Recall', 'F1']],
}).T
rule_vs_model

### Work through the "Should We Use AI?" checklist

Discuss with your group, then write your own answers in the cell below.

1. **Baseline:** Does the ML model clearly beat the simple rule (and the do-nothing baseline from Step 2)? By how much, and on which metric? Is that gain worth the added complexity?
2. **Cost of errors:** Using your Step 1 and Step 4 answers, what does a missed case cost vs. a false alarm? Does the model's error pattern match what the clinic can tolerate?
3. **Human in the loop:** Should the model make the final call, or only *prioritize* patients for a nurse to review? Which decisions need a human?
4. **Data & fairness:** This data covers one specific group of women, and some measurements are recorded as `0`. Would you trust this model on the clinic's actual patients? Who might it work worse for?
5. **Monitoring:** What would you track after launch to know if the model is going stale (Week 5)?

### ✅ Your recommendation
Choose one and justify it in 4–6 sentences using **specific numbers** from your evaluation:

- **Deploy the model** as the screening tool
- **Deploy with human review** (model prioritizes, a person decides)
- **Use the simple rule** instead of ML
- **Don't deploy yet** (and say what would need to change)

**Your answers (checklist):**

1. Baseline:

2. Cost of errors:

3. Human in the loop:

4. Data & fairness:

5. Monitoring:

**Your recommendation:**

### 📤 To submit for the in-class activity
Upload to the **IC-5 module**: a screenshot of your **Step 8 rule-vs-model table** and your written **recommendation** above.

---
# Assignment 5 — Evaluate, Tune, and Defend Your Model

Use the **same Pima diabetes data** from Step 0 (`X_train_y`, `X_test_y`, `y_train_y`, `y_test_y`) and the helpers we defined (`evaluate_model`, `compare_models`, `plot_comparison`, `threshold_table`). Run all earlier cells first.

Write **your own** thoughts in the answer cells. Answers that just restate the numbers without explaining *why* won't get full credit. Submit the completed notebook with all outputs visible.

---
## Task 1 — Add a fourth model: KNN (compare fairly)

KNN classifies a patient by looking at the *k* most similar patients in the training data. Because it works on **distances**, features on big scales (like `Insulin`) can drown out small ones (like `DiabetesPedigreeFunction`), so we **scale the features first** using a *pipeline*.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# A pipeline = scale the features, THEN run KNN (scaling is learned from training data only)
knn_y = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_y.fit(X_train_y, y_train_y)

# Add it to the comparison
models_yours['KNN (k=5)'] = knn_y

results_all = compare_models(models_yours, X_train_y, y_train_y, X_test_y, y_test_y)
display(results_all)
plot_comparison(results_all, 'All four models — Diabetes data')

**Task 1 answer.** Which of the four models would you pick if the clinic's priority is **catching as many diabetes cases as possible**, and which if the priority is **avoiding false alarms**? Cite the specific metrics that support each choice.

**Your answer:**

## Task 2 — Diagnose overfitting in KNN

For KNN, the complexity knob is `n_neighbors` (k). A **small k** (like 1) follows the training data very closely; a **large k** smooths things out. Plot train vs. cross-validated F1 across different values of k.

In [ ]:
k_values = range(1, 31)
train_f1_knn, cv_f1_knn = [], []

for k in k_values:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_train_y, y_train_y)
    train_f1_knn.append(f1_score(y_train_y, knn.predict(X_train_y)))
    cv_f1_knn.append(cross_val_score(knn, X_train_y, y_train_y, cv=5, scoring='f1').mean())

plt.figure(figsize=(7, 4))
plt.plot(k_values, train_f1_knn, marker='o', label='Training F1')
plt.plot(k_values, cv_f1_knn, marker='o', label='Cross-validated F1')
plt.xlabel('k (number of neighbors)')
plt.ylabel('F1 score')
plt.title('KNN: train vs. cross-validated F1')
plt.legend()
plt.show()

**Task 2 answer.**
(a) At k = 1, how do the training and cross-validated scores compare, and why does a 1-neighbor model overfit?
(b) Roughly what value of k looks best, and how did you decide?
(c) Is *small k* or *large k* the more "complex" model? Explain in your own words how this connects to tree depth from Step 5.

**Your answer:**

## Task 3 — Tune a model the honest way

Pick **Random Forest or KNN** and tune it with `GridSearchCV` (training data only!). Fill in your own grid in the cell below, run it, then report the **before vs. after** on the test set.

In [ ]:
# TODO: choose ONE model to tune by leaving it as-is or swapping the commented lines
from sklearn.ensemble import RandomForestClassifier

model_to_tune = RandomForestClassifier(random_state=42)
param_grid_task3 = {
    'n_estimators': [50, 100, 200],     # TODO: edit these values
    'max_depth': [3, 5, 8, None],       # TODO: edit these values
}

# --- or, to tune KNN instead, use these two lines ---
# model_to_tune = make_pipeline(StandardScaler(), KNeighborsClassifier())
# param_grid_task3 = {'kneighborsclassifier__n_neighbors': [3, 5, 7, 11, 15, 21]}   # TODO: edit

# TODO: choose your scoring metric: 'f1', 'recall', or 'precision' (and be ready to justify it)
grid_task3 = GridSearchCV(model_to_tune, param_grid_task3, cv=5, scoring='f1')
grid_task3.fit(X_train_y, y_train_y)

print('Best settings:', grid_task3.best_params_)
print(f'Best cross-validated score: {grid_task3.best_score_:.3f}')

# Before vs. after on the test set (we touch the test set once, here)
untuned = model_to_tune.fit(X_train_y, y_train_y)
tuned = grid_task3.best_estimator_
pd.DataFrame({
    'Untuned (defaults)': evaluate_model(untuned, X_train_y, y_train_y, X_test_y, y_test_y),
    'Tuned': evaluate_model(tuned, X_train_y, y_train_y, X_test_y, y_test_y),
}).T

**Task 3 answer.**
(a) Which scoring metric did you choose for tuning, and why does that choice fit the clinic's situation?
(b) Did tuning help on the test set? Did it change the overfit gap? If it barely helped, what does that suggest?

**Your answer:**

## Task 4 — Choose a threshold using costs

Take the **best model from your Task 1 or 3 results** and pick a decision threshold that minimizes total cost. Choose your own cost ratio and justify it.

In [ ]:
# TODO: put your chosen model here (it must be fitted and support predict_proba)
best_model = tuned          # e.g., tuned, forest_y, knn_y, model_yours, ...

probs_best = best_model.predict_proba(X_test_y)[:, 1]
table_task4 = threshold_table(y_test_y, probs_best)

# TODO: choose YOUR cost ratio and be ready to justify it
cost_fn = 5
cost_fp = 1

table_task4['total_cost'] = table_task4['missed (FN)'] * cost_fn + table_task4['false alarms (FP)'] * cost_fp
display(table_task4)

best_t4 = table_task4['total_cost'].idxmin()
print(f'\nLowest-cost threshold: {best_t4}')

ConfusionMatrixDisplay.from_predictions(
    y_test_y, (probs_best >= best_t4).astype(int),
    display_labels=['No diabetes', 'Diabetes'], cmap='Blues'
)
plt.title(f'Confusion matrix at threshold = {best_t4}')
plt.show()

**Task 4 answer.**
(a) What cost ratio did you choose and why is it realistic for this clinic?
(b) Compared with the default 0.5 threshold, how many more cases did you catch and how many more false alarms did you accept?
(c) We chose the threshold by looking at the **test set**. Explain why that's a problem in a real project, and describe what you'd do instead.

**Your answer:**

## Task 5 — Final reflection (your own words)

In 6–10 sentences, answer:

1. Across everything you ran, **what is your final recommendation** for the clinic (model, threshold, and whether a human is involved)? Support it with specific metrics, not just accuracy.
2. Give **two reasons** the test-set results here might be more optimistic than what the clinic would see in real life (think: test-set size, single split, how the data was collected, who is in it).
3. What is **one thing you would do next** to improve or better validate the model?

**Your answer:**